# Dependencies and Data

In [1]:
import pandas as pd
import geopandas as gpd
import linref as lr

In [2]:
LRS =   lr.LRS(key_col=['NLFID', 'chain'], loc_col='COUNTY_LOG_NBR', beg_col='CTL_BEGIN_', end_col='CTL_END_NB', geom_col='geometry', geom_m_col='geometry_m', closed='left_mod')
lr.set_default_lrs(LRS)

In [3]:
# Load linear events with geometry
roads =   gpd.read_file('testing/data/roadways.json').explode().rename(columns={'NLF_ID': 'NLFID'})
crashes = gpd.read_file('testing/data/crashes.json')

Skipping field INTERSECTION_LEG_ID: unsupported OGR type: 5


# Preparing LRS Data

In [4]:
# Add M values and chaining
roads = roads.lr.add_geom_m()
roads = roads.lr.add_chaining(replace=True)
crashes = roads.lr.impute_keys(crashes)
crashes = crashes[crashes.lr.valid_events]

/home/tshihadah/code/linref/linref/ext/base.py:1224: UserWarning: 62 rows contain missing key values after imputation.
  warnings.warn(msg, UserWarning)


In [5]:
roads['CRASH_YR_FIRST'] = roads.lr.relate(crashes)['CRASH_YR'].first()
roads['CRASH_YR_LAST']  = roads.lr.relate(crashes)['CRASH_YR'].last()
roads['CRASH_YR_LIST']  = roads.lr.relate(crashes)['CRASH_YR'].list()
roads['CRASH_YR_SET']   = roads.lr.relate(crashes)['CRASH_YR'].set()
roads['CRASH_YR_SUM']   = roads.lr.relate(crashes)['CRASH_YR'].sum()
roads['CRASH_YR_MEAN']  = roads.lr.relate(crashes)['CRASH_YR'].mean()
roads['CRASH_YR_MODE']  = roads.lr.relate(crashes)['CRASH_YR'].mode()
roads['CRASH_YR_COUNT'] = roads.lr.relate(crashes).count()
roads[['CRASH_YR_FIRST', 'CRASH_YR_LAST', 'CRASH_YR_LIST', 'CRASH_YR_SET', 'CRASH_YR_SUM', 'CRASH_YR_MEAN', 'CRASH_YR_MODE', 'CRASH_YR_COUNT']].sample(10)

,CRASH_YR_FIRST,CRASH_YR_LAST,CRASH_YR_LIST,CRASH_YR_SET,CRASH_YR_SUM,CRASH_YR_MEAN,CRASH_YR_MODE,CRASH_YR_COUNT
4829,NaN,NaN,[],{},0.0,0.0,NaN,0.0
2404,NaN,NaN,[],{},0.0,0.0,NaN,0.0
330,NaN,NaN,[],{},0.0,0.0,NaN,0.0
4362,NaN,NaN,[],{},0.0,0.0,NaN,0.0
724,NaN,NaN,[],{},0.0,0.0,NaN,0.0
3594,2019.0,2019.0,[2019],{2019},2019.0,2019.0,2019.0,1.0
2988,NaN,NaN,[],{},0.0,0.0,NaN,0.0
3027,NaN,NaN,[],{},0.0,0.0,NaN,0.0
3902,NaN,NaN,[],{},0.0,0.0,NaN,0.0
1987,NaN,NaN,[],{},0.0,0.0,NaN,0.0


In [6]:
roads.shape

(5110, 113)

# Dissolving Linear Data

In [7]:
# Perform dissolve
dissolved = roads.lr.dissolve()

# Resegmenting Linear Data at Regular Intervals

In [8]:
# Resegment the dissolved data at 0.5 mile intervals
segments = dissolved.lr.resegment(length=0.5, fill='balance')

# Integrate Multiple Sets of Linear Events

In [9]:
# Create additional resegmented linear events for example
segments_1 = dissolved.lr.resegment(length=0.5, fill='balance').sample(frac=0.5, random_state=1)
segments_2 = dissolved.lr.resegment(length=0.3, fill='balance').sample(frac=0.5, random_state=1)
# Integrate the resegmented linear events
integrated = lr.integrate([segments_1, segments_2])
# Cut new geometry at integrated locations
integrated = integrated.lr.cut_from(dissolved)

ValueError: Length of values (3872) does not match length of index (1392)

# Project Point Events onto the LRS

In [ ]:
# Project crash points onto the dissolved linear events
projected = dissolved.lr.project(crashes.drop(columns=['COUNTY_LOG_NBR', 'NLFID']))
# Interpolate new geometry at projected locations along roadway events
projected = projected.lr.interpolate_from(dissolved)